# Transfer Curves → Dashboard

Procesa curvas de transferencia desde HDF5, extrae V_th, genera CSV para el dashboard y registra en SQLite.

**Flujo:**
1. Seleccionar archivo HDF5
2. Extraer V_th con ajuste lineal global (√|I_DS| vs V_GS)
3. Visualizar ajustes para verificar calidad
4. Guardar CSV en `data/processed/`
5. Registrar en SQLite (memoria + referencias + análisis)

**Nota:** El dashboard permite ajustar individualmente los rangos de V_th por dispositivo.

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

def find_project_root(marker='src'):
    cwd = Path(os.getcwd()).resolve()
    for parent in [cwd, *cwd.parents]:
        if (parent / marker).is_dir():
            return parent
    raise FileNotFoundError(f"No se encontro project root con '{marker}'")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.readers.keithley_dean import list_measurements, read_curve, read_all_curves
from src.plotting import plot_transfer
from src.analysis import extract_vth_for_transfers
from src import db

db.init_db()
print('Project root:', PROJECT_ROOT)
print('Database initialized at', db.DB_PATH)

In [ ]:
data_folder = PROJECT_ROOT / 'data' / 'raw'

print('Archivos HDF5 disponibles:')
hdf5_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.hdf5')])
for f in hdf5_files:
    print(f'  {f}')

# === MODIFICAR: Seleccionar archivo HDF5 ===
hdf5_name = '2026-10-02_test.hdf5'
hdf5_path = PROJECT_ROOT / 'data' / 'raw' / hdf5_name

print('\n' + '='*80)
print(f'Archivo seleccionado: {hdf5_name}')
print(f'Ruta: {hdf5_path}')
print(f'Existe: {hdf5_path.exists()}')

if not hdf5_path.exists():
    raise FileNotFoundError(f'Archivo no encontrado: {hdf5_path}')

In [ ]:
measurement_list_dic = list_measurements(hdf5_path)
transfer_measurements = [m['name'] for m in measurement_list_dic if m['mode'] == 'transfer']

print(f'Transfer measurements ({len(transfer_measurements)}):')
for m in transfer_measurements:
    print(f'  {m}')

## Parámetros de ajuste global

Define el rango para el ajuste lineal √|I_DS| vs V_GS.

- `v_min_range`: (min, max) del límite inferior de la ventana
- `v_max_range`: (min, max) del límite superior de la ventana

El algoritmo busca la mejor combinación (v_min, v_max) que maximice R².

In [ ]:
# === MODIFICAR: Parámetros de ajuste ===
V_MIN_RANGE = (-0.40, -0.38)
V_MAX_RANGE = (-0.06, -0.00)

print(f'v_min_range: {V_MIN_RANGE} V')
print(f'v_max_range: {V_MAX_RANGE} V')

In [ ]:
df_vth = extract_vth_for_transfers(
    hdf5_path,
    transfer_measurements,
    v_min_range=V_MIN_RANGE,
    v_max_range=V_MAX_RANGE,
)

print(f'V_th extraído para {len(df_vth)} mediciones')
display(df_vth)

## Visualización del ajuste lineal

Gráficas de √|I_DS| vs V_GS con el ajuste lineal para verificar la calidad del fit.

In [ ]:
def plot_linearized_grid(hdf5_path, df_vth, ncols=3, figsize_per=(5, 4)):
    n = len(df_vth)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0]*ncols, figsize_per[1]*nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else np.array([axes])
    
    for ax, row in zip(axes_flat, df_vth.itertuples()):
        m_name = row.measurement
        v_min_w, v_max_w = row._4, row._5
        try:
            curves = read_all_curves(hdf5_path, m_name)
            i_max = max(range(len(curves)), key=lambda i: abs(curves[i]['v_ds'][0]))
            curve = curves[i_max]
            v_gs_fwd, i_ds_fwd = read_curve(hdf5_path, m_name, curve['curve_name'], fwd_only=True)
            sqrt_ids = np.sqrt(np.abs(i_ds_fwd))
            mask = (v_gs_fwd >= v_min_w) & (v_gs_fwd <= v_max_w)
            slope, intercept, r_value, _, _ = linregress(v_gs_fwd[mask], sqrt_ids[mask])
            v_th = -intercept / slope
            
            ax.plot(v_gs_fwd, sqrt_ids, 'rs-', ms=3, label=r'$\sqrt{|I_{DS}|}$')
            x_fit = np.array([v_min_w, v_max_w])
            y_fit = slope * x_fit + intercept
            ax.plot(x_fit, y_fit, 'g--', lw=2, label=f'fit ($V_{{th}}$={v_th:.3f} V, $R^2$={r_value**2:.4f})')
            ax.axvline(v_min_w, color='gray', ls=':', lw=0.8)
            ax.axvline(v_max_w, color='gray', ls=':', lw=0.8)
            ax.set_xlabel(r'$V_{GS}$ (V)')
            ax.set_ylabel(r'$\sqrt{|I_{DS}|}$ ($\sqrt{A}$)')
            ax.set_title(m_name, fontsize=9)
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error: {e}', transform=ax.transAxes, ha='center')
    
    for ax in axes_flat[len(df_vth):]:
        ax.set_visible(False)
    plt.tight_layout()
    return fig

if len(df_vth) > 0:
    fig = plot_linearized_grid(hdf5_path, df_vth)
    plt.show()
else:
    print('No hay datos de V_th para graficar')

## Guardar CSV en data/processed/

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

hdf5_stem = Path(hdf5_name).stem
csv_filename = f'transfer_vth_{hdf5_stem}.csv'
csv_path = processed_dir / csv_filename

df_vth.to_csv(csv_path, index=False)
print(f'CSV guardado en: {csv_path}')
print(f'Filas: {len(df_vth)}')
print(f'Columnas: {list(df_vth.columns)}')

## Registrar en SQLite

Crea o actualiza memoria en el dashboard con:
- Metadata del análisis
- Referencia al HDF5
- Referencia al CSV
- Resultado del análisis con parámetros

In [ ]:
def find_or_create_memory(hdf5_name, hdf5_rel_path, analysis_type, params):
    title = f'{analysis_type}: {hdf5_name}'
    hdf5_stem = Path(hdf5_name).stem
    
    # Buscar memoria existente
    existing = db.list_memories(search=hdf5_stem)
    for mem in existing:
        if mem['title'] == title:
            return mem['id'], False
    
    # Extraer fecha del nombre del HDF5 si es posible
    date_match = hdf5_stem[:10]
    try:
        datetime.strptime(date_match, '%Y-%m-%d')
        analysis_date = date_match
    except:
        analysis_date = datetime.now().strftime('%Y-%m-%d')
    
    # Extraer sustrato del nombre si es posible
    substrate = ''
    if 'kapton' in hdf5_name.lower():
        substrate = 'Kapton'
    elif 'si' in hdf5_name.lower():
        substrate = 'Si/SiO2'
    
    # Crear nueva memoria
    memory_id = db.create_memory(
        title=title,
        type='experiment',
        subtype='session',
        date=analysis_date,
        description=f'{analysis_type} analysis from {hdf5_name}',
        substrate=substrate,
        process='electrical measurement',
        custom_fields={
            'hdf5_file': hdf5_name,
            'analysis_date': datetime.now().strftime('%Y-%m-%d'),
            'n_measurements': str(len(transfer_measurements)),
        }
    )
    
    # Agregar referencia al HDF5
    db.add_reference(memory_id, 'hdf5', hdf5_rel_path)
    
    return memory_id, True

# Buscar o crear memoria
hdf5_rel_path = f'data/raw/{hdf5_name}'
memory_id, was_created = find_or_create_memory(
    hdf5_name, hdf5_rel_path, 'Transfer',
    {'v_min_range': list(V_MIN_RANGE), 'v_max_range': list(V_MAX_RANGE)}
)

print(f'Memory ID: {memory_id}')
print(f'Created: {was_created}')
print(f'Title: Transfer: {hdf5_name}')

In [ ]:
# Agregar referencia al CSV
csv_rel_path = f'data/processed/{csv_filename}'
db.add_reference(memory_id, 'csv', csv_rel_path)
print(f'CSV reference added: {csv_rel_path}')

# Guardar resultado del análisis
analysis_id = db.save_analysis_result(
    memory_id=memory_id,
    function_name='extract_vth_for_transfers',
    params={
        'v_min_range': list(V_MIN_RANGE),
        'v_max_range': list(V_MAX_RANGE),
    },
    results_csv_path=csv_rel_path,
)

print(f'Analysis result saved with ID: {analysis_id}')

## Resumen

✅ **Procesamiento completado:**
- HDF5: `{hdf5_name}`
- Mediciones: {len(transfer_measurements)} transfer curves
- V_th extraído con rangos: v_min={V_MIN_RANGE}, v_max={V_MAX_RANGE}
- CSV guardado: `{csv_rel_path}`
- Memoria registrada en SQLite: ID {memory_id}

## Siguientes pasos en el dashboard

1. Abrir el dashboard: `python dashboard/app.py`
2. Buscar la memoria "Transfer: {hdf5_name}"
3. Visualizar gráficas de V_th
4. Ajustar individualmente los rangos de V_th por dispositivo (override)
5. Exportar resultados si es necesario